# YgoMaster-PackBuilder — YgoMaster Exporter Tests

Test offline per `build_shop_export()`, `build_pack_odds_export()` e `build_main_shop_settings_patch()`.

Copre mapping rarità, pack type Selection, booster 3/7/11, Configuration A, assenza della garanzia R nel singolo pack, garanzia x10 SR/UR, pity UR, prezzi, cover, JSON round-trip ed errori principali.


## 1. Import e helper

In [1]:
from copy import deepcopy
import json

from ygomaster_packbuilder.domain import (
    Card, OpeningPolicy, PackDefinition, PackType, ProjectDefinition,
    Rarity, RarityDistribution, SourcePackRef, SourceProvider,
    TenPackGuarantee, UltraRarePity,
)
from ygomaster_packbuilder.exporters.ygomaster import (
    build_main_shop_settings_patch,
    build_pack_odds_export,
    build_shop_export,
)
from ygomaster_packbuilder.registry import get_pack_shop_id


def make_card(name, ydk_id, master_duel_id, rarity):
    return Card(
        name=name,
        ydk_id=ydk_id,
        source_rarity=rarity.value,
        master_duel_id=master_duel_id,
        rarity=rarity,
    )


def make_pack(key, name, description, pack_type, booster_size, cards, policy="configuration-a"):
    expected = {
        PackType.PERSONAL_MAIN: 100,
        PackType.PERSONAL_MINI: 20,
        PackType.SHARED_STAPLES: 200,
    }[pack_type]

    return PackDefinition(
        key=key,
        name=name,
        description=description,
        type=pack_type,
        source=SourcePackRef(
            provider=SourceProvider.YGOPRODECK,
            external_id=key,
        ),
        expected_card_count=expected,
        booster_size=booster_size,
        opening_policy_key=policy,
        cards=cards,
    )


def configuration_a():
    return OpeningPolicy(
        key="configuration-a",
        normal_slot_distribution=RarityDistribution(55.0, 35.0, 7.5, 2.5),
        ten_pack_guarantee=TenPackGuarantee(
            True,
            RarityDistribution(0.0, 0.0, 80.0, 20.0),
        ),
        ultra_rare_pity=UltraRarePity(
            True,
            RarityDistribution(0.0, 0.0, 0.0, 100.0),
        ),
    )

print("Helpers ready")

Helpers ready


## 2. Progetto di test

In [2]:
main_pack = make_pack(
    "main-pack", "Main Pack", "Main pack description",
    PackType.PERSONAL_MAIN, 7,
    [
        make_card("Normal", 1, 1001, Rarity.N),
        make_card("Rare", 2, 1002, Rarity.R),
        make_card("Super", 3, 1003, Rarity.SR),
        make_card("Ultra", 4, 1004, Rarity.UR),
    ],
)

mini_pack = make_pack(
    "mini-pack", "Mini Pack", "",
    PackType.PERSONAL_MINI, 3,
    [
        make_card("Mini Normal", 11, 1011, Rarity.N),
        make_card("Mini Ultra", 12, 1012, Rarity.UR),
    ],
)

shared_pack = make_pack(
    "shared-pack", "Shared Staples", "Shared description",
    PackType.SHARED_STAPLES, 11,
    [
        make_card("Shared Rare", 21, 1021, Rarity.R),
        make_card("Shared Super", 22, 1022, Rarity.SR),
    ],
)

project = ProjectDefinition(
    name="Exporter Test",
    packs=[main_pack, mini_pack, shared_pack],
    opening_policies=[configuration_a()],
)

pack_ids = {
    "main-pack": 9_000_000,
    "mini-pack": 9_000_001,
    "shared-pack": 9_000_002,
}

print("Project ready")

Project ready


## 3. Main Shop settings patch

In [3]:
assert build_main_shop_settings_patch() == {
    "PerPackRarities": True,
}
print("Settings patch OK")

Settings patch OK


## 4. Shop export

In [4]:
shop_export = build_shop_export(project, pack_ids)
pack_shop = shop_export["PackShop"]

assert set(pack_shop) == {
    str(get_pack_shop_id(9_000_000)),
    str(get_pack_shop_id(9_000_001)),
    str(get_pack_shop_id(9_000_002)),
}

main_shop_id = str(get_pack_shop_id(9_000_000))
mini_shop_id = str(get_pack_shop_id(9_000_001))
shared_shop_id = str(get_pack_shop_id(9_000_002))

item = pack_shop[main_shop_id]

assert item["packId"] == 9_000_000
assert item["packType"] == 2
assert item["secretType"] == 0
assert item["pack_card_num"] == 7
assert item["price"] == 100
assert item["priceX10"] == 1000

assert item["cardList"] == {
    "1001": 1,
    "1002": 2,
    "1003": 3,
    "1004": 4,
}

assert item["iconMrk"] == 1004
assert item["iconType"] == 1
assert item["iconData"] == "1004"
assert item["preview"] == [{"type": 1, "mrk": 1004}]

assert pack_shop[mini_shop_id]["descShortTextId"] == "Mini Pack"
assert pack_shop[mini_shop_id]["pack_card_num"] == 3
assert pack_shop[shared_shop_id]["pack_card_num"] == 11

assert json.loads(json.dumps(shop_export)) == shop_export

print("Shop export OK")

Shop export OK


## 5. Odds export — Configuration A

In [5]:
odds_export = build_pack_odds_export(project, pack_ids)

assert len(odds_export) == 3
assert [x["packShopIds"][0] for x in odds_export] == [
    get_pack_shop_id(9_000_000),
    get_pack_shop_id(9_000_001),
    get_pack_shop_id(9_000_002),
]

for odds in odds_export:
    assert odds["gachaType"] == 1

rows = odds_export[0]["cardRateList"]

assert rows[0] == {
    "start_num": 1,
    "end_num": 7,
    "standard": False,
    "rate": {
        "1": {"rate": "55.00"},
        "2": {"rate": "35.00"},
        "3": {"rate": "7.50"},
        "4": {"rate": "2.50"},
    },
}

# Nessuna guaranteed-R row per il singolo pack.
assert [
    row for row in rows
    if row.get("settle_rare_min") == 2
] == []

assert rows[1] == {
    "start_num": 7,
    "end_num": 7,
    "standard": False,
    "settle_rare_min": 3,
    "settle_rare_max": 4,
    "rate": {
        "3": {"rate": "80.00"},
        "4": {"rate": "20.00"},
    },
}

assert rows[2] == {
    "start_num": 7,
    "end_num": 7,
    "standard": False,
    "settle_rare_min": 4,
    "settle_rare_max": 4,
    "rate": {
        "4": {"rate": "100.00"},
    },
}

# L'ultimo slot segue il booster size.
assert odds_export[0]["cardRateList"][1]["start_num"] == 7
assert odds_export[1]["cardRateList"][1]["start_num"] == 3
assert odds_export[2]["cardRateList"][1]["start_num"] == 11

assert json.loads(json.dumps(odds_export)) == odds_export

print("Configuration A odds OK")

Configuration A odds OK


## 6. Premiere rates

In [6]:
expected = [
    {
        "rare": [4, 3],
        "rate": {
            "3": "1.00",
            "2": "9.00",
            "1": "90.00",
        },
    },
    {
        "rare": [2, 1],
        "rate": {
            "2": "10.00",
            "1": "90.00",
        },
    },
]

for odds in odds_export:
    assert odds["premiereRateList"] == expected

print("Premiere rates OK")

Premiere rates OK


## 7. Guarantee disabilitate

In [7]:
no_guarantees = OpeningPolicy(
    key="no-guarantees",
    normal_slot_distribution=RarityDistribution(55.0, 35.0, 7.5, 2.5),
    ten_pack_guarantee=TenPackGuarantee(
        False,
        RarityDistribution(0.0, 0.0, 0.0, 0.0),
    ),
    ultra_rare_pity=UltraRarePity(
        False,
        RarityDistribution(0.0, 0.0, 0.0, 0.0),
    ),
)

pack = deepcopy(main_pack)
pack.opening_policy_key = "no-guarantees"

result = build_pack_odds_export(
    ProjectDefinition(
        name="No Guarantees",
        packs=[pack],
        opening_policies=[no_guarantees],
    ),
    {"main-pack": 9_000_000},
)

assert len(result[0]["cardRateList"]) == 1
assert "settle_rare_min" not in result[0]["cardRateList"][0]

print("Disabled guarantees omitted")

Disabled guarantees omitted


## 8. Pack ID mancante

In [8]:
for builder in (
    build_shop_export,
    build_pack_odds_export,
):
    try:
        builder(project, {"main-pack": 9_000_000})
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            f"{builder.__name__} avrebbe dovuto fallire per pack ID mancante."
        )

Expected ValueError: No YgoMaster pack ID assigned to pack 'mini-pack'.
Expected ValueError: No YgoMaster pack ID assigned to pack 'mini-pack'.


## 9. Opening policy sconosciuta

In [9]:
bad = deepcopy(project)
bad.packs[0].opening_policy_key = "missing-policy"

try:
    build_pack_odds_export(bad, pack_ids)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Opening policy sconosciuta non rifiutata.")

Expected ValueError: Pack 'main-pack' references unknown opening policy 'missing-policy'.


## 10. Master Duel ID irrisolto

In [10]:
bad = deepcopy(project)
bad.packs[0].cards[0].master_duel_id = None

try:
    build_shop_export(bad, pack_ids)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Master Duel ID irrisolto non rifiutato.")

Expected ValueError: Card 'Normal' in pack 'main-pack' has no Master Duel ID.


## 11. Rarity irrisolta

In [11]:
bad = deepcopy(project)
bad.packs[0].cards[0].rarity = None

try:
    build_shop_export(bad, pack_ids)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Rarity irrisolta non rifiutata.")

Expected ValueError: Card 'Normal' in pack 'main-pack' has no rarity.


## 12. Master Duel ID duplicato

In [12]:
bad = deepcopy(project)
bad.packs[0].cards[1].master_duel_id = (
    bad.packs[0].cards[0].master_duel_id
)

try:
    build_shop_export(bad, pack_ids)
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Master Duel ID duplicato non rifiutato.")

Expected ValueError: Pack 'main-pack' contains duplicate Master Duel card ID 1001.


## 13. Pack senza cover utilizzabile

In [13]:
empty_pack = make_pack(
    "empty-pack", "Empty", "",
    PackType.PERSONAL_MAIN, 7, [],
)

empty_project = ProjectDefinition(
    name="Empty",
    packs=[empty_pack],
    opening_policies=[configuration_a()],
)

try:
    build_shop_export(
        empty_project,
        {"empty-pack": 9_000_000},
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Pack senza cover non rifiutato.")

Expected ValueError: Pack 'empty-pack' has no usable cards for its shop icon.
